In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, TimeDistributed, RepeatVector
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import mean_squared_error, mean_absolute_error

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn import preprocessing


In [ ]:
col_names = ["unit", "time_cycles", "op_set1", "op_set2", "op_set3"] + [f'sensor{i}' for i in range(1, 22)]
base_path = "/content/drive/MyDrive/Colab/CMAPSSData/"

df_train = pd.read_csv(base_path + "train_FD001.txt", sep=r"\s+", header=None, names=col_names)
df_test = pd.read_csv(base_path + "test_FD001.txt", sep=r"\s+", header=None, names=col_names)
df_rul = pd.read_csv(base_path + "RUL_FD001.txt", sep=r"\s+", header=None, names=["RUL"])

In [ ]:
train_unit_100 = df_train[df_train["unit"] == 100]
test_unit_100 = df_test[df_test["unit"] == 100]
sensors_to_plot = ['sensor1', 'sensor3', 'sensor20']

fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(10, 8), sharex=True)

for i, sensor in enumerate(sensors_to_plot):
    axes[i].plot(train_unit_100["time_cycles"], train_unit_100[sensor], color='blue')
    axes[i].set_ylabel(sensor + " Value")
    axes[i].set_title(sensor)
    axes[i].grid(True)

axes[-1].set_xlabel("Time Cycles")
plt.tight_layout()
plt.show()

In [ ]:
df_train['RUL'] = df_train.groupby('unit')['time_cycles'].transform(lambda x: x.max() - x)
#We know max lifetime, so we can use x.max()-x in time cycles for find RUL with grouping by units
df_train

In [ ]:
exclude_cols = [f'sensor{i}' for i in [1, 5, 6, 10, 16, 18, 19]] #Unnecessary(stable) sensors were removed
feature_cols = df_train.filter(regex='sensor').columns.difference(exclude_cols) #op_set not important for FD001

scaler = preprocessing.StandardScaler() #instead of minmax scaler
df_train[feature_cols] = scaler.fit_transform(df_train[feature_cols])
df_test[feature_cols] = scaler.transform(df_test[feature_cols]) # Old fit was used for prevent data leak

In [ ]:
windowing_length = 30

def create_train_sequences(df, seq_len):
    sequences, targets = [], []
    for unit in df['unit'].unique():
        engine_data = df[df['unit'] == unit]
        features = engine_data[feature_cols].values
        rul_values = engine_data['RUL'].values

        for i in range(len(features) - seq_len + 1):
            sequences.append(features[i : i + seq_len])
            targets.append(rul_values[i + seq_len - 1]) #RUL values append to targets

    return np.array(sequences), np.array(targets)

def get_last_test_windows(df, seq_len):
    last_windows = []
    for unit in df['unit'].unique():
        engine_data = df[df['unit'] == unit]
        features = engine_data[feature_cols].values
        last_windows.append(features[-seq_len:]) #last 30 window important for evaluate RUL

    return np.array(last_windows)

In [ ]:
X_train, y_train = create_train_sequences(df_train, windowing_length)
X_test = get_last_test_windows(df_test, windowing_length)

y_test = df_rul.values

print(f"X_train: {X_train.shape}")
print(f"Y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_test: {y_test.shape}")

In [ ]:
MAX_RUL = 125
y_train_clipped = np.clip(y_train, a_min=None, a_max=MAX_RUL)
y_test_clipped = np.clip(y_test, a_min=None, a_max=MAX_RUL)

timesteps = X_train.shape[1]
n_features = X_train.shape[2]

inputs = Input(shape=(timesteps, n_features))
encoded = LSTM(64, return_sequences=True)(inputs)
encoded = LSTM(32, return_sequences=False)(encoded)

decoded = RepeatVector(timesteps)(encoded)
decoded = LSTM(32, return_sequences=True)(decoded)
decoded = LSTM(64, return_sequences=True)(decoded)
outputs = TimeDistributed(Dense(n_features))(decoded)

autoencoder = Model(inputs, outputs)
autoencoder.compile(optimizer='adam', loss='mse')

autoencoder.fit(X_train, X_train, epochs=40, batch_size=64, validation_split=0.1, verbose=1)

encoder_model = Model(inputs, encoded)
X_train_encoded = encoder_model.predict(X_train)
X_test_encoded = encoder_model.predict(X_test)

rul_model = Sequential([
    Dense(64, activation='relu', input_shape=(32,)),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1)
])

rul_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5)

history = rul_model.fit(
    X_train_encoded, y_train_clipped,
    epochs=60,
    batch_size=64,
    validation_split=0.1,
    shuffle=True,
    callbacks=[early_stop, reduce_lr]
)

y_pred = rul_model.predict(X_test_encoded)

rmse = np.sqrt(mean_squared_error(y_test_clipped, y_pred))
mae = mean_absolute_error(y_test_clipped, y_pred)

print("-" * 30)
print(f"Final Test RMSE: {rmse:.2f}")
print(f"Final Test MAE : {mae:.2f}")
print("-" * 30)

y_test_1d = y_test_clipped.flatten()
y_pred_1d = y_pred.flatten()

In [ ]:
sort_idx = np.argsort(y_test_1d)[::-1]

y_test_sorted = y_test_1d[sort_idx]
y_pred_sorted = y_pred_1d[sort_idx]

plt.figure(figsize=(14, 6))
plt.plot(y_test_sorted, label='Actual RUL', color='blue', linewidth=2.5)
plt.plot(y_pred_sorted, label='Predicted RUL', color='red', alpha=0.8, linestyle='--')

plt.axhline(y=125, color='green', linestyle=':', label='125 Limit')

plt.title("Test Data: Actual vs Predicted RUL", fontsize=14)
plt.xlabel("Test Engines", fontsize=12)
plt.ylabel("Remain RUL", fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.5)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 8))
plt.scatter(y_test_1d, y_pred_1d, alpha=0.7, color='purple', edgecolor='k')

plt.plot([0, max(y_test_1d)], [0, max(y_test_1d)], color='black', linestyle='--', label='Y=X')

plt.title("Actual RUL vs Predicted RUL", fontsize=14)
plt.xlabel("Actual RUL", fontsize=12)
plt.ylabel("Model Predict", fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.5)
plt.tight_layout()
plt.show()